# Tendon vibration — P03, P04 and P06

**Only two participants have vibration recordings.** P05's 24-07 session is polarity x lidocaine;
its log does not mention vibration at all. So this is a two-participant comparison, not three.

## The thing that decides how this must be read

The vibrator was on a **different muscle and a different side** in each participant:

| | vibrator on | arm | conditions recorded |
|---|---|---|---|
| **P03** | wrist **extensor** | **left** | baseline, applied-but-off, ON (+ a 5 mA-step repeat for 30 Hz) |
| **P04** | **flexor** tendon | **right** | baseline, OFF, ON flexors, ON **extensors** (a check, vibrator moved) |

So P03's "VIB ON" and P04's "VIB ON extensors" are **not the same condition**. Comparing them
directly would compare left-extensor vibration against right-extensor vibration in different
people.

**What is comparable is the CHANGE at the vibrated site**, measured against that participant's own
baseline. Every bar below is a % of the same participant's baseline at the same intensity;
100 % = unchanged. The vibrated muscle is marked with `*`.

## One more caveat, from the logs

P03's "baseline" block was recorded at 16:31 with the **vibrator already attached** (applied
~15:45, switched off), and ~1.5 h into the session. P04's baseline is at 14:22, before the
vibrator was involved at all. So P03's baseline is closer to P04's *VIB OFF* than to P04's
baseline.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       vibration_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       check_thresholds, mt_stamp, warn_if_stale,
                       fig_threshold_grid,
                       fig_detection_row, fig_detection_grid,
                       fig_group_summary, fig_group_panels,
                       condition_table, snr_verdict, fig_noise_check)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D03 = "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/"
D04 = "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/"
D06 = "tSCS_CHUV_data/29-09-2026/"    # P06, 29-09-2026. That folder also
# holds a copy of the whole 28-09 session; only the 20260929 files are this one.

RUN = {   # (subject, condition) -> csv.  Set PROTOCOL to "burst" or "arcex".
 "burst": {
   ("P03", "Baseline"):        D03 + "Burst_autosave_20260715_164431_737ms.csv",
   ("P03", "VIB off"):         D03 + "Burst_autosave_20260715_165831_999ms.csv",
   # P03's VIB ON burst was recorded twice, and this is the second: the session log calls it
   # "redoing for step of 5 mA". It steps 5 mA like the baseline and VIB off blocks, so every
   # muscle is read at its exact threshold and none has to move a step. The first take (17:06,
   # below) stepped 10 mA, which left Flex. dig (L) and Flex. carpi rad. (R) without their
   # 35 and 45 mA. This one sits 16 min further into the vibration block.
   ("P03", "VIB ON"):          D03 + "Burst_autosave_20260715_172234_038ms.csv",   # 17:22, step 5
 # ("P03", "VIB ON"):          D03 + "Burst_autosave_20260715_170629_569ms.csv",   # 17:06, step 10
   ("P04", "Baseline"):        D04 + "Burst_autosave_20260716_142311_232ms.csv",
   ("P04", "VIB off"):         D04 + "Burst_autosave_20260716_144116_893ms.csv",
   ("P04", "VIB ON"):          D04 + "Burst_autosave_20260716_145100_665ms.csv",   # flexors
   ("P04", "VIB ON extensors"):D04 + "Burst_autosave_20260716_152821_805ms.csv",
   # P06, cathodic ("negative" in its log), vibrator on the left extensors as in P03
   ("P06", "Baseline"):        D06 + "Burst_autosave_20260929_155626_234ms.csv",   # 15:56
   ("P06", "VIB off"):         D06 + "Burst_autosave_20260929_163809_190ms.csv",   # 16:38
   ("P06", "VIB ON"):          D06 + "Burst_autosave_20260929_175220_421ms.csv",   # 17:52
 },
 "arcex": {
   ("P03", "Baseline"):        D03 + "Modulated_autosave_20260715_164630_593ms.csv",
   ("P03", "VIB off"):         D03 + "Modulated_autosave_20260715_170032_049ms.csv",
   ("P03", "VIB ON"):          D03 + "Modulated_autosave_20260715_171433_605ms.csv",
   ("P04", "Baseline"):        D04 + "Modulated_autosave_20260716_142536_785ms.csv",
   ("P04", "VIB off"):         D04 + "Modulated_autosave_20260716_144439_044ms.csv",
   ("P04", "VIB ON"):          D04 + "Modulated_autosave_20260716_145845_958ms.csv",  # flexors
   ("P04", "VIB ON extensors"):D04 + "Modulated_autosave_20260716_153556_059ms.csv",
   ("P06", "Baseline"):        D06 + "Modulated_autosave_20260929_160545_334ms.csv",  # 16:05
   ("P06", "VIB off"):         D06 + "Modulated_autosave_20260929_164440_099ms.csv",  # 16:44
   ("P06", "VIB ON"):          D06 + "Modulated_autosave_20260929_170027_460ms.csv",  # 17:00
 },
}
PROTOCOLS  = ["burst", "arcex"]       # both, side by side - no switch to flip
NAME       = {"burst": "30 Hz", "arcex": "ARC-EX"}
SUBJECTS   = ["P03", "P04", "P06"]
BLOCKS     = ["Baseline", "VIB off", "VIB ON", "VIB ON extensors"]   # everything recorded
CONTROL    = "VIB off"                                # what the analysis compares against
CONDITIONS = [CONTROL, "VIB ON", "VIB ON extensors"]  # the bars; the last one is P04 only
# The threshold of each participant comes from their CONTROL recording and all their blocks are
# read at it, muscle by muscle - the same rule as P06_tendon_vibration_polarity.ipynb.
#
# VIB off is the control because it is the only block that isolates the vibration: the vibrator
# is strapped on and silent, so it shares the setup, the electrodes and the elapsed time with
# VIB ON and differs in one thing. Baseline is 30-60 min earlier with nothing attached, and in
# P06 the step from Baseline to VIB off was larger than anything VIB ON added. Baseline stays in
# BLOCKS so sections 2 and 3 still show it. Set CONTROL = "Baseline" to put it back.

# ======== the signal-to-noise gate ========
SNR_GATE = None    # None = draw every reading. 2.0 = leave out anything under twice its noise.
SNR_SHOW = 2.0     # still MARKS anything under this, and lists it for the noise check
# The gate divides by the PRE-STIMULUS noise, and the vibrator is a motor on the skin over the
# muscle it is strapped to: it raises that electrode's noise floor several fold, so the gate
# fails that muscle even where its response is unchanged. Section 3 measures it.
# ==========================================
VIBRATED   = {"P03": "Ext. digitorum (L)",    # the muscle the vibrator sat on
              "P04": "Flex. digitorum (R)",
              "P06": "Ext. digitorum (L)"}    # left extensors, the same site as P03
SITE       = {"P03": "left wrist extensor",   # ...in words, for the panel titles
              "P04": "right flexor tendon",
              "P06": "left wrist extensor"}
# the same four muscles as the baseline comparison
MUSCLES = ["Flex. digitorum (L)", "Ext. digitorum (L)",      # left arm, then right
           "Flex. digitorum (R)", "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

# one panel per participant AND protocol, so both protocols are in every figure
PANELS = [f"{s} · {NAME[p]}" for s in SUBJECTS for p in PROTOCOLS]
runs   = {(f"{s} · {NAME[p]}", cond): RUN[p][(s, cond)]
          for s in SUBJECTS for p in PROTOCOLS for cond in BLOCKS if (s, cond) in RUN[p]}
# the vibrated muscle and the site belong to the participant, so they carry over to both panels
VIB_PANEL  = {f"{s} · {NAME[p]}": VIBRATED[s] for s in SUBJECTS for p in PROTOCOLS}
SITE_PANEL = {f"{s} · {NAME[p]}": SITE[s] for s in SUBJECTS for p in PROTOCOLS}

for pan in PANELS:
    for cond in BLOCKS:
        if (pan, cond) not in runs:
            continue
        f = runs[(pan, cond)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:16s} {cond:18s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA"
              + ("   <- control" if cond == CONTROL else ""))

## 2 · The motor thresholds you chose

The threshold is picked in each participant's **`VIB off`** recording and used for **all** of that
participant's blocks, muscle by muscle. Nothing crosses between participants, and the `VIB ON`
picks are not used.

Two grids: **what is saved for each recording**, read straight from `results/<session>/mt_*.csv`,
and below the numbers **what the analysis uses** — one marked sweep per panel, that participant's
`VIB off` threshold, drawn on every one of their rows. That is the trace every number in section 4
comes from.

In [ ]:
CHECK = {f"{pan} · {cond}": runs[(pan, cond)] for pan in PANELS for cond in BLOCKS
         if (pan, cond) in runs}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title="what is saved for each recording")

In [ ]:
# the control recording of each participant sets the intensity for all of their blocks
MT = {pan: thresholds_for(runs[(pan, CONTROL)], MUSCLES, consecutive=2, **KW)
      for pan in PANELS}
for pan in PANELS:
    print(f"{pan:16s} {CONTROL}  ({threshold_source(runs[(pan, CONTROL)])})  "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[pan].items())))

# the same grid, with the line on the sweep each block is analysed at
USED = {f"{pan} · {cond}": MT[pan] for pan in PANELS for cond in BLOCKS
        if (pan, cond) in runs}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED, share_gain=True,
                       title=f"the motor threshold used: each panel's {CONTROL}")

MT_STAMP = mt_stamp(runs)

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — **each at the threshold you picked in that
recording** (the red trace in section 2 — so this checks your own picks, block by block). The
10 pulses are re-aligned on their own onset, with the max (v) and min (^) the peak-to-peak is
made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

> Section 4 is different on purpose: there every condition is measured at the **Baseline**
> threshold, so vibration cannot move the intensity as well as the response. A muscle drops out
> of section 4 when that baseline mA was never delivered in the later block — P03's VIB blocks
> step 10 mA where its baseline steps 5, so 35 and 45 mA do not exist there.

### If a peak you can see is not being taken

`TRY` changes the settings **for these figures only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

# the same grid as section 2, showing how the peaks are found AT THE INTENSITY SECTION 4
# MEASURES - every block of a panel at that panel's VIB off threshold, not at its own pick
DET_MT = {f"{pan} · {cond}": MT[pan] for pan in PANELS for cond in BLOCKS
          if (pan, cond) in runs}
fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=f"peak detection at each panel's {CONTROL} threshold", **TRY)

### What the vibrator does to the noise — check this before reading section 4

The vibrator is a motor strapped over one muscle, and its mechanical noise lands in that
electrode. Below is the pre-stimulus noise floor of each muscle in each block, in mV, relative to
that participant's control block.

Any criterion that divides by noise fails the vibrated muscle even where its response is
unchanged, which is why `SNR_GATE` is off. Judge that muscle on its **millivolts** in the
per-muscle figures.

In [ ]:
from functions.burst import noise_p2p as _noise, burst_p2p as _b

print(f"{'panel':16s}{'block':18s}" + "".join(f"{m.split(' (')[0][:8] + m[-3:]:>17s}"
                                               for m in MUSCLES))
for pan in PANELS:
    ref = {}
    for b in BLOCKS:
        if (pan, b) not in runs:
            continue
        meta_, t_, sig_ = load_run(runs[(pan, b)])
        chans = [c for c in sig_ if c != "Trigger A"]
        res_ = _b(meta_, t_, sig_, chans, **{**KW, "min_snr": None, "max_edge_frac": None})
        nb_ = _noise(t_, sig_, chans, win_len_ms=res_["win"][0][1] - res_["win"][0][0])
        amps_ = list(res_["amps"])
        row = ""
        for m in MUSCLES:
            ch = next((c for c in chans if pretty(c) == m), None)
            th = MT[pan].get(m)
            if ch is None or not th or th not in amps_:
                row += f"{'-':>17s}"; continue
            v = np.asarray(nb_[ch], float)
            v = float(v[amps_.index(th)]) if v.ndim else float(v)
            ref.setdefault(m, v)
            row += f"{v:9.4f} x{v / ref[m]:4.1f}{'*' if VIB_PANEL.get(pan) == m else ' '}"
        print(f"{pan:16s}{b:18s}{row}")
print("\n* = that participant's vibrated muscle.  x = noise relative to their first block shown.")

## 4 · The effect of vibration

**Three bars per muscle.** Grey is `VIB off` at 100 %. Beside it, each vibration block measured two
ways, both as a % of `VIB off` measured the same way: its **1st pulse**, and the **mean of all 10
pulses**. They are not the same number — a manipulation can move one without the other — so say
which you are quoting.

`*` marks the vibrated muscle. **`VIB ON` is not the same site in every participant**, so read the
panel title with it:

| participant | `VIB ON` (orange) | `VIB ON extensors` (blue) |
|---|---|---|
| **P03** | left wrist **extensor** | — |
| **P04** | right **flexor** tendon | **extensors**, a second block its log calls *"just to double check what happens in the extensors"*; the side is not recorded |
| **P06** | left wrist **extensor** | — |

So the orange bars are the extensor site in P03 and P06 and the **flexor** site in P04, and the
blue bars exist for P04 alone. A real effect shows up at the muscle marked `*` and not at the
others.

**The noise under the vibrator is not the noise elsewhere.** Section 3's table shows the vibrated
muscle's noise floor rising several fold once the vibrator is attached, which is why `SNR_GATE` is
off: a reading is marked `?` when it is under `SNR_SHOW` × its noise rather than removed, and the
noise panels below let you decide whether it is a small response or nothing.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])

tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base=CONTROL, match="common", **KW)

# one figure, the control at 100 % and each vibration block measured both ways
VIB_BLOCKS = [c for c in CONDITIONS if c != CONTROL]
BARS = [CONTROL] + [f"{c} · {w}" for c in VIB_BLOCKS for w in ("1st pulse", "mean 1-10")]
BAR_COLOUR = {CONTROL: "#8C8C8C",
              "VIB ON · 1st pulse": "#F0A868", "VIB ON · mean 1-10": "#E67E22",
              "VIB ON extensors · 1st pulse": "#8FBCD4", "VIB ON extensors · mean 1-10": "#2E6F95"}
# "VIB ON" is NOT the same site in every participant - P04's is the right flexor tendon, P03's
# and P06's the left wrist extensor - so the panel title names it. The blue bars are P04's
# SECOND block only, the one the log calls "VIB ON (Extensors)".
both = {}
for pan in PANELS:
    for m in MUSCLES:
        if (pan, CONTROL, m) in tab:
            both[(pan, CONTROL, m)] = dict(tab[(pan, CONTROL, m)], pct=100.0)
        for c_ in VIB_BLOCKS:
            v = tab.get((pan, c_, m))
            if v:
                both[(pan, f"{c_} · 1st pulse", m)] = dict(v, pct=v.get("pct"))
                both[(pan, f"{c_} · mean 1-10", m)] = dict(v, pct=v.get("pct_all"))

fig_vibration(both, PANELS, MUSCLES, BARS, colours=BAR_COLOUR,
              vibrated=VIB_PANEL, site=SITE_PANEL,
              min_snr_ratio=SNR_GATE, zero_if_absent=True,
              ylabel=f"% of {CONTROL}", ylim=(0, 200),
              title=f"Tendon vibration — % of {CONTROL}: 1st pulse and whole train",
              save=None)

### The numbers, so they can be checked

In [ ]:
print(f"{'panel':16s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{c[:12] + ' 10p':>18s}{c[:12] + ' 1st':>18s}" for c in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, c_, m)]["amp"] for c_ in CONDITIONS if (pan, c_, m) in tab), None)
        if amp is None:
            continue
        cells = []
        for c_ in CONDITIONS:
            v, why = snr_verdict(tab, (pan, c_, m), CONTROL, SNR_GATE)
            if why:
                cells += [why, ""]
            elif v is None:
                cells += ["-", "-"]
            else:
                mk = " ?" if v.get("snr", 9e9) < SNR_SHOW else "  "
                cells += [f"{v['pct_all']:.0f} %" + mk, f"{v['pct']:.0f} %" + mk]
        star = " *" if VIB_PANEL.get(pan) == m else "  "
        print(f"{pan:16s}{m + star:22s}{amp:5g}" + "".join(f"{c_:>18s}" for c_ in cells))
print(f"\n10p = mean of all 10 pulses   1st = the 1st pulse only, both as % of {CONTROL}")
print(f"?  = under {SNR_SHOW:g}x its own noise - drawn, but check it in the noise panels below")

### Before vs during vibration, one figure per muscle

The 1st pulse, the 2nd, and the mean of pulses 2–10, with the EMG underneath in matching
colours.

In [ ]:
# ======== settings ========
SHOW_MUSCLES = MUSCLES
BARS_COND    = [c for c in CONDITIONS if any((pan, c) in runs for pan in PANELS)]
COND_COLOUR  = {CONTROL: "#8C8C8C", "VIB ON": "#E67E22", "VIB ON extensors": "#2E6F95"}
LINES        = {CONTROL: "-", "VIB ON": "-", "VIB ON extensors": (0, (4, 2))}
# ==========================

MT_cond = {(pan, cond): {m: tab[(pan, cond, m)]["amp"] for m in SHOW_MUSCLES
                          if (pan, cond, m) in tab}
           for s in SUBJECTS for cond in BARS_COND if (pan, cond) in runs}

for m in SHOW_MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(BARS_COND), steps=0,
             colours=COND_COLOUR, names={c: c for c in BARS_COND}, linestyles=LINES,
             min_snr_ratio=SNR_GATE, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

### 4d · The three participants together

The same numbers pooled: for each muscle the **mean over participants** with **SD**, and every
participant as a dot in its bar's colour, for the whole train and for the 1st pulse.

Read it knowing the vibrated muscle differs between them — `Ext. digitorum (L)` for P03 and P06,
`Flex. digitorum (R)` for P04 — so no single muscle is "the vibrated one" across the group. The
per-participant figure above is where the vibration effect lives; this one shows whether the
unvibrated muscles drift the same way in all three.

In [ ]:
BASE_MAP = {c: CONTROL for c in CONDITIONS}

for key, lab, ttl in (("pct_all", f"% of {CONTROL} (mean 1-10)",
                       f"Tendon vibration — whole train, % of {CONTROL}"),
                      ("pct", f"% of {CONTROL} (1st pulse)",
                       f"Tendon vibration — 1st pulse, % of {CONTROL}")):
    fig_group_summary(tab, PANELS, MUSCLES, CONDITIONS, base=BASE_MAP, key=key,
                      colours=COND_COLOUR, min_snr_ratio=SNR_GATE, err="sd",
                      reference=(CONTROL, 100),
                      ylabel=lab, ref=100, ylim=(0, 200), title=ttl,
                      save=None)

### Is it noise, or is there nothing there? — decide by eye

For every reading marked `?`, the response windows beside windows of the same length from the
90 ms **before** the stimulus, on one scale. Alike means noise. A deflection at a consistent
latency that the pre-stimulus windows do not have means a response, however small the ratio.

In [ ]:
ONLY = None     # None = every marked reading; or ("P06 · 30 Hz", "VIB ON", "Ext. digitorum (L)")

flagged = [(pan, c_, m) for pan in PANELS for c_ in CONDITIONS for m in MUSCLES
           if (pan, c_, m) in tab and tab[(pan, c_, m)].get("snr", 9e9) < SNR_SHOW]
todo = [ONLY] if ONLY else flagged
print(f"{len(flagged)} reading(s) under {SNR_SHOW:g}x noise"
      + ("  (the gate is OFF - all of them are in the figures)" if SNR_GATE is None else "") + "\n")
for pan, c_, m in todo:
    fig_noise_check(runs[(pan, c_)], m, tab[(pan, c_, m)].get("amp"),
                    title=f"{pan} · {c_} · {m}", **KW)

## 5 · Change a threshold

**Six cells** — each participant's `VIB off` recording in each protocol, which is the control
everything in section 4 is measured against. Move the slider, press **Set to slider**, then
**Save**, and re-run from section 2.

Nothing else needs picking: `Baseline` and the `VIB ON` blocks set no intensity. To pick one
anyway, call `repick(("P03 · 30 Hz", "Baseline"))` in a new cell — it changes nothing
downstream.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not the baseline, and not
    whatever is already saved (that is the orange trace)."""
    csv = RUN[key] if isinstance(RUN, dict) and key in RUN else runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    print("   the analysis uses this participant's BASELINE thresholds: "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[key[0]].items())
                      if m in MUSCLES))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P03 · 30 Hz', 'VIB off'))

In [ ]:
repick(('P03 · ARC-EX', 'VIB off'))

In [ ]:
repick(('P04 · 30 Hz', 'VIB off'))

In [ ]:
repick(('P04 · ARC-EX', 'VIB off'))

In [ ]:
repick(('P06 · 30 Hz', 'VIB off'))

In [ ]:
repick(('P06 · ARC-EX', 'VIB off'))